MODELO DE REGRESSÃO

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

caminho_dados = 'https://raw.githubusercontent.com/AnaRaquelCafe/POSTECH_AI_SCIENTIST/refs/heads/main/Base%20de%20dados%20Tech%20Challenge/desafio_nps_fase_1.csv'

dados = pd.read_csv(caminho_dados)

dados.head(10)

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9
5,6,35,Sudeste,75,50006,568.76,6,36.58,3,4,5,41.82,2,2,3,0.9,0,5,1.5
6,7,37,Sudeste,68,50007,41.29,3,99.62,6,8,3,35.83,3,3,4,1.4,0,6,0.6
7,8,60,Sul,37,50008,428.76,4,29.54,10,11,5,44.50,1,0,2,0.0,0,2,4.1
8,9,40,Sul,60,50009,121.56,3,91.95,6,6,3,24.88,2,1,9,6.2,0,3,0.8
9,10,51,Sudeste,70,50010,411.01,6,37.47,3,9,2,30.59,1,0,7,2.7,0,2,4.2


In [11]:
# 1. Definindo a Variável Alvo (Y)
Y = dados['nps_score']

# 2. Selecionando apenas as Variáveis de Entrada (X) operacionais relevantes conforme seu EDA
colunas_preditoras = [
    'delivery_delay_days',
    'complaints_count',
    'customer_service_contacts',
    'resolution_time_days'
]

X = dados[colunas_preditoras]

# 3. Adicionando a constante (intercepto/alfa) para a Regressão Linear do Statsmodels
X_com_const = sm.add_constant(X)

# Verificando as 5 primeiras linhas da matriz X pronta
X_com_const.head()

,const,delivery_delay_days,complaints_count,customer_service_contacts,resolution_time_days
0,1.0,2,3,0,4
1,1.0,4,3,0,10
2,1.0,1,7,4,5
3,1.0,2,4,1,11
4,1.0,1,3,1,0


In [12]:
# Dividindo a base: 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(
    X_com_const, 
    Y, 
    test_size=0.20, 
    random_state=42
)

# Conferindo o tamanho de cada base
print(f"Clientes na base de Treino: {len(X_train)}")
print(f"Clientes na base de Teste: {len(X_test)}")

Clientes na base de Treino: 2000
Clientes na base de Teste: 500


In [13]:
# 1. Definindo e treinando o modelo OLS com os dados de treino
modelo_nps = sm.OLS(y_train, X_train).fit()

# 2. Exibindo o resumo estatístico detalhado
print(modelo_nps.summary())

                            OLS Regression Results                            
Dep. Variable:              nps_score   R-squared:                       0.556
Model:                            OLS   Adj. R-squared:                  0.555
Method:                 Least Squares   F-statistic:                     623.5
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        12:41:22   Log-Likelihood:                -3866.4
No. Observations:                2000   AIC:                             7743.
Df Residuals:                    1995   BIC:                             7771.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                 

In [17]:
# 1. Realizando as predições no conjunto de TESTE
y_pred = modelo_nps.predict(X_test)

# 2. Calculando as métricas de erro MAE e RMSE
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Erro Médio Absoluto (MAE): {mae:.2f} pontos no NPS")
print(f"Erro Quadrático Médio (MSE): {mse:.2f} pontos no NPS")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} pontos no NPS")

Erro Médio Absoluto (MAE): 1.32 pontos no NPS
Erro Quadrático Médio (MSE): 2.80 pontos no NPS
Raiz do Erro Quadrático Médio (RMSE): 1.67 pontos no NPS


MODELO DE CLASSIFICAÇÃO

In [18]:
# Criando a coluna 'is_detractor': 1 se nps_score < 7, senão 0
dados['is_detractor'] = (dados['nps_score'] < 7.0).astype(int)

# Verificando a distribuição da nova variável alvo
print(dados['is_detractor'].value_counts())
print("\nProporção:")
print(dados['is_detractor'].value_counts(normalize=True) * 100)

is_detractor
1    2109
0     391
Name: count, dtype: int64

Proporção:
is_detractor
1    84.36
0    15.64
Name: proportion, dtype: float64


In [19]:
from sklearn.linear_model import LogisticRegression

# 1. Definindo o novo Y (variável binária) e o X (as 4 variáveis operacionais)
Y_class = dados['is_detractor']
X_class = dados[['delivery_delay_days', 'complaints_count', 'customer_service_contacts', 'resolution_time_days']]

# 2. Dividindo novamente em Treino (80%) e Teste (20%)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, Y_class, test_size=0.20, random_state=42
)

# 3. Criando e treinando o modelo de Regressão Logística
modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_train_c, y_train_c)

# 4. Fazendo as previsões na base de Teste
y_pred_class = modelo_logistico.predict(X_test_c)

print("Modelo treinado e previsões geradas com sucesso!")

Modelo treinado e previsões geradas com sucesso!


In [20]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Gerando a Matriz de Confusão
matriz = confusion_matrix(y_test_c, y_pred_class)

# 2. Exibindo os resultados de forma clara
print("=== MATRIZ DE CONFUSÃO ===")
print(matriz)
print("\n=== RELATÓRIO DE DESEMPENHO ===")
print(classification_report(y_test_c, y_pred_class, target_names=['Não-Detrator (0)', 'Detrator (1)']))

=== MATRIZ DE CONFUSÃO ===
[[ 24  49]
 [ 20 407]]

=== RELATÓRIO DE DESEMPENHO ===
                  precision    recall  f1-score   support

Não-Detrator (0)       0.55      0.33      0.41        73
    Detrator (1)       0.89      0.95      0.92       427

        accuracy                           0.86       500
       macro avg       0.72      0.64      0.67       500
    weighted avg       0.84      0.86      0.85       500

